# ToolCall-200M — Clean SentencePiece Tokenizer Training

This notebook performs **only tokenizer training**.

## Final artifact

```text
toolcall_spm_32k.model
```

Use the native `sentencepiece` library to tokenize the full pretraining corpus into integer token shards.

## Pipeline

1. Configure storage and optional Hugging Face authentication.
2. Stream a representative tokenizer corpus.
3. Train a 32k SentencePiece BPE tokenizer.
4. Validate encoding, decoding, token IDs, and special tokens.
5. Save and optionally upload the artifacts.

## Default corpus mixture

The default corpus is **1.5 GB**, chosen to be representative while remaining realistic for Colab.

| Source | Share | Purpose |
|---|---:|---|
| FineWeb-Edu | 45% | Clean general English |
| CodeParrot Clean | 20% | Code, identifiers, paths, function names |
| APIs.guru OpenAPI Directory | 20% | API schemas, JSON/YAML, parameters |
| xLAM Function Calling 60k | 10% | Function-calling syntax and tool schemas |
| Synthetic ToolCall text | 5% | Project-specific special tokens and output format |

The xLAM dataset is optional. If access is unavailable, its allocation is transferred to synthetic structured text.

> SentencePiece training is CPU-bound. A T4 is not required for this notebook.

In [ ]:
!pip -q install -U "datasets>=3.0.0" "huggingface_hub>=0.25.0" "sentencepiece>=0.2.0" "tqdm>=4.66.0" "pyyaml>=6.0"

## 1. Storage and optional Hugging Face authentication

In [ ]:
USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/toolcall_200m/tokenizer"
else:
    PROJECT_DIR = "/content/toolcall_200m/tokenizer"

print("PROJECT_DIR:", PROJECT_DIR)

In [ ]:
# Required only for gated datasets such as Salesforce/xlam-function-calling-60k.
# Recommended: add HF_TOKEN in Colab Settings -> Secrets.

USE_XLAM = True

if USE_XLAM:
    from huggingface_hub import login

    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None

    if hf_token:
        login(token=hf_token, add_to_git_credential=False)
        print("Logged into Hugging Face using the HF_TOKEN Colab secret.")
    else:
        print(
            "HF_TOKEN was not found. The notebook will try xLAM later and "
            "fall back safely if access is unavailable."
        )

## 2. Configuration

In [ ]:
import hashlib
import json
import os
import random
import re
import shutil
import subprocess
import time
import unicodedata
from pathlib import Path
from typing import Any, Dict, Iterable, Iterator

from tqdm.auto import tqdm

PROJECT_DIR = Path(PROJECT_DIR)
RAW_DIR = PROJECT_DIR / "raw"
OUTPUT_DIR = PROJECT_DIR / "output"
CORPUS_PATH = RAW_DIR / "tokenizer_corpus.txt"
MODEL_PREFIX = OUTPUT_DIR / "toolcall_spm_32k"

RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1.5 GB is a practical default. Reduce to 1 GB if network/storage is slow.
TOTAL_TARGET_BYTES = int(1.5 * 1024**3)

SOURCE_WEIGHTS = {
    "fineweb_edu": 0.45,
    "codeparrot": 0.20,
    "openapi": 0.20,
    "xlam": 0.10,
    "synthetic": 0.05,
}
SOURCE_BUDGETS = {
    name: int(TOTAL_TARGET_BYTES * weight)
    for name, weight in SOURCE_WEIGHTS.items()
}

VOCAB_SIZE = 32_000
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

SPECIAL_TOKENS = [
    "<|system|>",
    "<|user|>",
    "<|assistant|>",
    "<|tool_schema|>",
    "<|tool_call|>",
    "<|tool_result|>",
    "<|json|>",
    "<|end|>",
    "<|endoftext|>",
]

print(f"Corpus target: {TOTAL_TARGET_BYTES / 1024**3:.2f} GB")
for name, budget in SOURCE_BUDGETS.items():
    print(f"{name:16s}: {budget / 1024**2:8.1f} MB")

## 3. Corpus utilities

In [ ]:
CONTROL_CHARS = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]")
EXCESS_BLANK_LINES = re.compile(r"\n{4,}")


def normalize_text(value: Any) -> str:
    if value is None:
        return ""
    text = value if isinstance(value, str) else str(value)
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = CONTROL_CHARS.sub("", text)
    text = EXCESS_BLANK_LINES.sub("\n\n\n", text)
    return text.strip()


def iter_text_chunks(text: str, max_chars: int = 8192, min_chars: int = 20) -> Iterator[str]:
    """Split documents into bounded pieces so SentencePiece does not skip long lines."""
    text = normalize_text(text)
    if not text:
        return

    paragraphs = re.split(r"\n\s*\n", text)
    buffer = ""

    for paragraph in paragraphs:
        paragraph = paragraph.strip()
        if not paragraph:
            continue

        pieces = [paragraph[i:i + max_chars] for i in range(0, len(paragraph), max_chars)]
        for piece in pieces:
            if len(buffer) + len(piece) + 2 <= max_chars:
                buffer = f"{buffer}\n{piece}".strip()
            else:
                if len(buffer) >= min_chars:
                    yield buffer
                buffer = piece

    if len(buffer) >= min_chars:
        yield buffer


def compact_json(value: Any) -> str:
    if isinstance(value, str):
        return value
    return json.dumps(value, ensure_ascii=False, separators=(",", ":"))


def write_source(output, source_name: str, documents: Iterable[str], byte_budget: int) -> Dict[str, int]:
    stats = {"bytes": 0, "documents": 0, "chunks": 0}
    progress = tqdm(total=byte_budget, unit="B", unit_scale=True, desc=source_name)

    for document in documents:
        document_had_chunk = False
        for chunk in iter_text_chunks(document):
            record = chunk + "\n<|endoftext|>\n"
            encoded_size = len(record.encode("utf-8"))

            if stats["bytes"] + encoded_size > byte_budget:
                progress.close()
                return stats

            output.write(record)
            stats["bytes"] += encoded_size
            stats["chunks"] += 1
            document_had_chunk = True
            progress.update(encoded_size)

        if document_had_chunk:
            stats["documents"] += 1

    progress.close()
    return stats

## 4. Dataset iterators

In [ ]:
from datasets import load_dataset


def iter_fineweb_edu() -> Iterator[str]:
    dataset = load_dataset(
        "HuggingFaceFW/fineweb-edu",
        name="sample-10BT",
        split="train",
        streaming=True,
    )
    for row in dataset:
        text = row.get("text")
        if text:
            yield text

In [ ]:
PERMISSIVE_LICENSES = {
    "mit", "apache-2.0", "bsd-2-clause", "bsd-3-clause",
    "isc", "mpl-2.0", "unlicense",
}


def iter_codeparrot() -> Iterator[str]:
    dataset = load_dataset(
        "codeparrot/codeparrot-clean",
        split="train",
        streaming=True,
    )

    for row in dataset:
        if bool(row.get("autogenerated", False)):
            continue

        license_name = str(row.get("license", "")).strip().lower()
        if license_name and license_name not in PERMISSIVE_LICENSES:
            continue

        content = row.get("content")
        if not content:
            continue

        repo_name = row.get("repo_name", "")
        path = row.get("path", "")
        yield f"# repository: {repo_name}\n# path: {path}\n{content}"

In [ ]:
OPENAPI_REPO_DIR = RAW_DIR / "openapi-directory"


def prepare_openapi_repository() -> None:
    if OPENAPI_REPO_DIR.exists():
        print("OpenAPI Directory already exists:", OPENAPI_REPO_DIR)
        return

    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/APIs-guru/openapi-directory.git",
            str(OPENAPI_REPO_DIR),
        ],
        check=True,
    )


def iter_openapi_specs() -> Iterator[str]:
    prepare_openapi_repository()
    api_root = OPENAPI_REPO_DIR / "APIs"
    if not api_root.exists():
        raise FileNotFoundError(f"Expected API directory not found: {api_root}")

    for path in api_root.rglob("*"):
        if path.suffix.lower() not in {".json", ".yaml", ".yml"}:
            continue
        try:
            content = path.read_text(encoding="utf-8", errors="ignore")
        except OSError:
            continue
        relative_path = path.relative_to(OPENAPI_REPO_DIR)
        yield f"# OpenAPI source: {relative_path}\n{content}"

In [ ]:
def iter_xlam() -> Iterator[str]:
    dataset = load_dataset(
        "Salesforce/xlam-function-calling-60k",
        split="train",
        token=True,
    )

    for row in dataset:
        query = row.get("query", "")
        tools = row.get("tools", "")
        answers = row.get("answers", "")

        yield (
            "<|user|>\n" + str(query) + "\n"
            "<|tool_schema|>\n" + compact_json(tools) + "\n"
            "<|tool_call|>\n" + compact_json(answers) + "\n"
            "<|end|>"
        )

In [ ]:
TOOL_CATALOG = {
    "calendar": {
        "create_event": ["title", "start_time", "end_time"],
        "search_events": ["query", "start_date", "end_date"],
        "update_event": ["event_id", "changes"],
    },
    "email": {
        "search_emails": ["query", "max_results"],
        "draft_email": ["recipient", "subject", "body"],
        "archive_emails": ["message_ids"],
    },
    "tasks": {
        "create_task": ["title", "deadline", "priority"],
        "list_tasks": ["status", "limit"],
        "complete_task": ["task_id"],
    },
    "files": {
        "search_files": ["query", "file_type"],
        "read_file": ["path"],
        "move_file": ["source_path", "destination_path"],
    },
    "spreadsheet": {
        "filter_rows": ["sheet", "filters"],
        "groupby_aggregate": ["sheet", "group_by", "metric", "aggregation"],
        "export_csv": ["sheet", "output_path"],
    },
}

USER_REQUESTS = [
    "Schedule the project review tomorrow at 3 PM.",
    "Find unread emails from HR about the internship.",
    "Create a high-priority task to submit the report by Friday.",
    "Search my files for the deployment checklist.",
    "Group spreadsheet expenses by category and sum the amount.",
    "Move the final report into the archive folder.",
    "Draft an email confirming that I can join next Monday.",
]


def value_for_parameter(name: str) -> Any:
    examples = {
        "title": "Project review",
        "start_time": "2026-07-20T15:00:00+05:30",
        "end_time": "2026-07-20T16:00:00+05:30",
        "query": "internship from HR",
        "start_date": "2026-07-01",
        "end_date": "2026-07-31",
        "event_id": "evt_1042",
        "changes": {"start_time": "2026-07-20T16:00:00+05:30"},
        "max_results": 10,
        "recipient": "hr@example.com",
        "subject": "Joining confirmation",
        "body": "I confirm that I can join next Monday.",
        "message_ids": ["msg_10", "msg_11"],
        "deadline": "2026-07-24",
        "priority": "high",
        "status": "open",
        "limit": 20,
        "task_id": "task_210",
        "file_type": "pdf",
        "path": "/documents/report.pdf",
        "source_path": "/documents/report.pdf",
        "destination_path": "/archive/report.pdf",
        "sheet": "expenses",
        "filters": {"status": "approved"},
        "group_by": "category",
        "metric": "amount",
        "aggregation": "sum",
        "output_path": "/exports/expenses.csv",
    }
    return examples.get(name, f"example_{name}")


def iter_synthetic_tool_calls() -> Iterator[str]:
    # Tokenizer-only supplement; not the final SFT dataset.
    while True:
        domain = random.choice(list(TOOL_CATALOG))
        action = random.choice(list(TOOL_CATALOG[domain]))
        parameters = TOOL_CATALOG[domain][action]
        tool_name = f"{domain}.{action}"

        properties = {
            name: {
                "type": (
                    "integer" if name in {"max_results", "limit"}
                    else "array" if name == "message_ids"
                    else "object" if name in {"changes", "filters"}
                    else "string"
                )
            }
            for name in parameters
        }

        schema = {
            "name": tool_name,
            "description": action.replace("_", " "),
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": parameters,
            },
        }

        output = {
            "decision": "call",
            "tool_name": tool_name,
            "arguments": {name: value_for_parameter(name) for name in parameters},
            "missing_required_fields": [],
        }

        yield (
            "<|user|>\n" + random.choice(USER_REQUESTS) + "\n"
            "<|tool_schema|>\n" + compact_json(schema) + "\n"
            "<|tool_call|>\n" + compact_json(output) + "\n"
            "<|end|>"
        )

## 5. Build the representative tokenizer corpus

In [ ]:
REBUILD_CORPUS = True

if CORPUS_PATH.exists() and not REBUILD_CORPUS:
    print("Using existing corpus:", CORPUS_PATH)
else:
    if CORPUS_PATH.exists():
        CORPUS_PATH.unlink()

    source_stats = {}
    synthetic_extra_budget = 0

    with CORPUS_PATH.open("w", encoding="utf-8", buffering=1024 * 1024) as output:
        source_stats["fineweb_edu"] = write_source(
            output, "FineWeb-Edu", iter_fineweb_edu(), SOURCE_BUDGETS["fineweb_edu"]
        )
        source_stats["codeparrot"] = write_source(
            output, "CodeParrot", iter_codeparrot(), SOURCE_BUDGETS["codeparrot"]
        )
        source_stats["openapi"] = write_source(
            output, "OpenAPI", iter_openapi_specs(), SOURCE_BUDGETS["openapi"]
        )

        if USE_XLAM:
            try:
                source_stats["xlam"] = write_source(
                    output, "xLAM", iter_xlam(), SOURCE_BUDGETS["xlam"]
                )
                shortfall = SOURCE_BUDGETS["xlam"] - source_stats["xlam"]["bytes"]
                synthetic_extra_budget += max(0, shortfall)
            except Exception as error:
                print("xLAM was skipped:", repr(error))
                print("Its allocation will be replaced with synthetic structured text.")
                source_stats["xlam"] = {"bytes": 0, "documents": 0, "chunks": 0}
                synthetic_extra_budget += SOURCE_BUDGETS["xlam"]
        else:
            source_stats["xlam"] = {"bytes": 0, "documents": 0, "chunks": 0}
            synthetic_extra_budget += SOURCE_BUDGETS["xlam"]

        synthetic_budget = SOURCE_BUDGETS["synthetic"] + synthetic_extra_budget
        source_stats["synthetic"] = write_source(
            output, "Synthetic ToolCall", iter_synthetic_tool_calls(), synthetic_budget
        )

    actual_size = CORPUS_PATH.stat().st_size
    print("\nCorpus complete")
    print("Path:", CORPUS_PATH)
    print(f"Size: {actual_size / 1024**3:.3f} GB")
    print(json.dumps(source_stats, indent=2))

## 6. Train the native 32k SentencePiece BPE tokenizer

In [ ]:
import sentencepiece as spm

if not CORPUS_PATH.exists() or CORPUS_PATH.stat().st_size == 0:
    raise FileNotFoundError("Tokenizer corpus is missing or empty.")

model_path = Path(str(MODEL_PREFIX) + ".model")
vocab_path = Path(str(MODEL_PREFIX) + ".vocab")

for path in (model_path, vocab_path):
    if path.exists():
        path.unlink()

start_time = time.time()

spm.SentencePieceTrainer.train(
    input=str(CORPUS_PATH),
    model_prefix=str(MODEL_PREFIX),
    vocab_size=VOCAB_SIZE,
    model_type="bpe",
    byte_fallback=True,
    character_coverage=0.9995,
    split_digits=False,
    normalization_rule_name="nfkc",
    remove_extra_whitespaces=False,
    allow_whitespace_only_pieces=True,
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
    pad_piece="<|pad|>",
    unk_piece="<|unk|>",
    bos_piece="<|bos|>",
    eos_piece="<|eos|>",
    user_defined_symbols=SPECIAL_TOKENS,
    input_sentence_size=5_000_000,
    shuffle_input_sentence=True,
    max_sentence_length=16_384,
    train_extremely_large_corpus=True,
    num_threads=max(1, os.cpu_count() or 2),
    random_seed_sentencepiece=RANDOM_SEED,
)

elapsed = time.time() - start_time
print(f"Training finished in {elapsed / 60:.1f} minutes")
print("Model:", model_path)
print("Vocabulary:", vocab_path)

## 7. Validate the native tokenizer — this is the source of truth

In [ ]:
sp = spm.SentencePieceProcessor(model_file=str(model_path))

TEST_STRINGS = [
    "calendar.create_event",
    "gmail.search_emails",
    "gradient_accumulation_steps",
    "Asia/Kolkata",
    "2026-06-25T15:30:00+05:30",
    '{"decision":"call","tool_name":"calendar.create_event","arguments":{"title":"Project sync"}}',
    (
        "<|user|>\nSchedule a meeting with Rahul tomorrow at 4 PM.\n"
        "<|tool_schema|>\n"
        '{"name":"calendar.create_event"}\n'
        "<|tool_call|>"
    ),
    "/home/user/projects/toolcall_200m/configs/train_config.yaml",
    "spreadsheet.groupby_aggregate",
    "missing_required_fields",
]

for text in TEST_STRINGS:
    ids = sp.encode(text, out_type=int)
    pieces = sp.encode(text, out_type=str)
    decoded = sp.decode(ids)

    if not ids:
        raise RuntimeError(f"Tokenizer returned zero tokens for: {text!r}")

    print("=" * 100)
    print(text)
    print("Token count:", len(ids))
    print("Round-trip exact:", decoded == text)
    print("Pieces:", pieces[:80])

In [ ]:
assert sp.vocab_size() == VOCAB_SIZE, (
    f"Expected vocabulary size {VOCAB_SIZE}, got {sp.vocab_size()}"
)

expected_core_ids = {
    "<|pad|>": 0,
    "<|unk|>": 1,
    "<|bos|>": 2,
    "<|eos|>": 3,
}

for piece, expected_id in expected_core_ids.items():
    actual_id = sp.piece_to_id(piece)
    assert actual_id == expected_id, f"{piece}: expected {expected_id}, got {actual_id}"

for token in SPECIAL_TOKENS:
    token_id = sp.piece_to_id(token)
    assert token_id != sp.unk_id(), f"Special token missing: {token}"
    print(f"{token:20s} -> {token_id}")

print("\nTokenizer validation passed.")

In [ ]:
# Optional comparison against GPT-2. This is diagnostic only.
!pip -q install -U transformers pandas

import pandas as pd
from transformers import AutoTokenizer

gpt2 = AutoTokenizer.from_pretrained("gpt2")
rows = []

for text in TEST_STRINGS:
    custom_count = len(sp.encode(text, out_type=int))
    gpt2_count = len(gpt2.encode(text, add_special_tokens=False))
    rows.append({
        "text": text[:100],
        "custom_spm_32k": custom_count,
        "gpt2": gpt2_count,
        "custom_vs_gpt2": round(custom_count / max(gpt2_count, 1), 3),
    })

display(pd.DataFrame(rows))

## 8. Save manifest, usage helper, and archive

In [ ]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

manifest = {
    "project": "ToolCall-200M",
    "tokenizer_type": "SentencePiece BPE",
    "vocab_size": VOCAB_SIZE,
    "byte_fallback": True,
    "split_digits": False,
    "normalization": "NFKC",
    "corpus_size_bytes": CORPUS_PATH.stat().st_size,
    "source_weights": SOURCE_WEIGHTS,
    "source_budgets_bytes": SOURCE_BUDGETS,
    "special_tokens": SPECIAL_TOKENS,
    "files": {"model": model_path.name, "vocab": vocab_path.name},
    "sha256": {
        "model": sha256_file(model_path),
        "vocab": sha256_file(vocab_path),
    },
    "usage": (
        "Use sentencepiece.SentencePieceProcessor(model_file=...) for "
        "pre-tokenization. The tokenizer must remain frozen after pretraining starts."
    ),
}

manifest_path = OUTPUT_DIR / "tokenizer_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

helper_path = OUTPUT_DIR / "tokenizer_usage.py"
helper_code = "\n".join([
    "from pathlib import Path",
    "import sentencepiece as spm",
    "",
    "class ToolCallTokenizer:",
    "    def __init__(self, model_path: str | Path):",
    "        self.processor = spm.SentencePieceProcessor(model_file=str(model_path))",
    "",
    "    @property",
    "    def vocab_size(self) -> int:",
    "        return self.processor.vocab_size()",
    "",
    "    def encode(self, text: str, add_bos: bool = False, add_eos: bool = True) -> list[int]:",
    "        ids = self.processor.encode(text, out_type=int)",
    "        if add_bos:",
    "            ids.insert(0, self.processor.bos_id())",
    "        if add_eos:",
    "            ids.append(self.processor.eos_id())",
    "        return ids",
    "",
    "    def decode(self, token_ids: list[int]) -> str:",
    "        return self.processor.decode(token_ids)",
])
helper_path.write_text(helper_code, encoding="utf-8")

archive_path = shutil.make_archive(
    str(OUTPUT_DIR / "toolcall_spm_32k_artifacts"),
    "zip",
    root_dir=OUTPUT_DIR,
)

print("Manifest:", manifest_path)
print("Usage helper:", helper_path)
print("Archive:", archive_path)

## 9. Optional Hugging Face Hub upload

This uploads the native SentencePiece artifacts. It deliberately does not create a misleading or untested Hugging Face tokenizer wrapper.

In [ ]:
# Uncomment after setting REPO_ID and authenticating.
# from huggingface_hub import create_repo, upload_folder
#
# REPO_ID = "your-username/toolcall-spm-32k"
# create_repo(REPO_ID, repo_type="model", exist_ok=True)
# upload_folder(
#     repo_id=REPO_ID,
#     repo_type="model",
#     folder_path=str(OUTPUT_DIR),
# )
# print("Uploaded:", REPO_ID)

# What happens next

This notebook stops after producing and validating the tokenizer.

The next pipeline stage is:

```text
raw pretraining datasets
    -> native SentencePiece encoding
    -> packed uint16 token shards
    -> ToolCall-200M pretraining
```

Because the vocabulary has 32,000 entries, token IDs fit in `uint16`. Freeze the tokenizer and record its SHA-256 hash before producing any pretraining shards.